In [ ]:
import os
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'

In [ ]:
import sys
sys.path.append('../')
sys.path.append('../Stretch2Relax')

In [ ]:
import MeshFEM, parallelism
import mesh, mesh_energy, py_newton_optimizer, viewer, param_utils
import parametrization, benchmark
import numpy as np

# PP's recorded run used one thread; MeshFEM's TBB pool is independent of OMP.
# parallelism.set_max_num_tbb_threads(1)

In [ ]:
import continuation_parametrization, flip_avoiding_step_length

In [ ]:
from Benchmark import helper_funcs
import extra_utils
import opt_utils

# Read Mesh and initialization

In [ ]:
m = helper_funcs.read_mesh('../../models/bird_small.msh.xz')

In [ ]:
uv = mesh_energy.NodalVars(m, 2)

In [ ]:
bdry_uv = helper_funcs.getBDdataOnNormalizedCircle(m)
uv_init = helper_funcs.tutteInitialization(m, bdry_uv)

In [ ]:
uv.setVars(uv_init.ravel())

# Prob Set up

In [ ]:
param = continuation_parametrization.symmetric_dirichlet_param(m, uv)
objectives = [param]

In [ ]:
# Construct parametrization energy and problem
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(uv, objectives)
opt = prob.optimizer()

In [ ]:
prob.initialFeasibleStepLengthComputer = flip_avoiding_step_length.FlipAvoidingStepLength(m.elements())
# PP's Armijo search starts at 0.95 * step_to_singularity. The line search
# applies the 0.95 below, so the feasibility limiter must return the unscaled step.
prob.initialFeasibleStepLengthComputer.backoffFactor = 1.0

In [ ]:
# prob.hessianShift = 1e-9
# prob.useRelativeHessianShift = True

param.elementHessianShift = 1e-6

In [ ]:
prob.objective()

In [ ]:
DEGREE = 0

In [ ]:
# opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
# opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 2
# opt.options.hessianProjectionController.startWithProjectionActive = False

opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()

In [ ]:
# benchmark.reset()
# param.setInterpolatedReference(0, uv_init.ravel())
# opt.options.hessianProjectionController.reset()
# prob.invalidateCachedHessian()
# opt.update_factorizations()
# print(prob.hessianWasProjected)
# benchmark.report()

## read l from file

In [ ]:
target_dir = 'linux_exp_data/match_CM_true_area-pp_linesearch'
# target_dir = 'linux_exp_data/cm_only_true_area'

In [ ]:
# Replay PP's continuation schedule; this isolates the per-step optimizer match.
l_arr = np.loadtxt(f'{target_dir}/fig12_a_pp_t.txt')
pp_energy = np.loadtxt(f'{target_dir}/fig12_a_pp_energy.txt')
pp_grad_norm = np.loadtxt(f'{target_dir}/fig12_a_pp_grad_norm.txt')

# PP optimize iteration

In [ ]:
linear_extrapolator = extra_utils.LinearExtrapolator()
backtrack_armijo_linesearch = opt_utils.BacktrackArmijoLineSearch()

In [ ]:
backtrack_armijo_linesearch.step_limiter = prob.initialFeasibleStepLengthComputer
backtrack_armijo_linesearch.max_alpha = np.inf

In [ ]:
stats_trace = []

In [ ]:
benchmark.reset()

# param.setInterpolatedReference(0, uv_init.ravel())
prob.setVars(uv_init.ravel())

for it, l in enumerate(l_arr):
    
    print('l: ', l)
    param.setInterpolatedReference(l, prob.getVars())
    
    prob.invalidateCachedHessian()
    # opt.options.hessianProjectionController.reset()
    opt.update_factorizations()
    
    # Match PP's CM_match() line search: start at 0.95 of the singular step,
    # then halve until Armijo with c = 0.2 is satisfied.
    x_list = opt_utils.newton_extrapolate(opt, linear_extrapolator, backtrack_armijo_linesearch,
                                    max_iters=1, verbose=True)
    # opt.options.niter = 1
    # opt.options.gradTol = 1e-6
    # opt.optimize()
    
    # eval prob at 1.0
    param.setInterpolatedReference(1.0)
    stats_trace.append({
        'iter': it,
        'l': float(l),
        'energy': prob.energy(),
        'grad_norm': np.linalg.norm(prob.gradient()),
    })
    # print(f'energy: {prob.energy()}; gradient_norm: {np.linalg.norm(prob.gradient())}')

benchmark.report()


# Debug

In [ ]:
for st in stats_trace:
    print(f"{st['iter']}    {st['l']}    {st['energy']}    {st['grad_norm']:.5e}")

In [ ]:
python_energy = np.array([st['energy'] for st in stats_trace])
python_grad_norm = np.array([st['grad_norm'] for st in stats_trace])

print('max |log10 energy difference|:', np.max(np.abs(np.log10(python_energy) - np.log10(pp_energy))))
print('max |log10 grad-norm difference|:', np.max(np.abs(np.log10(python_grad_norm) - np.log10(pp_grad_norm))))
print('final energy (Python, PP):', python_energy[-1], pp_energy[-1])
print('final grad norm (Python, PP):', python_grad_norm[-1], pp_grad_norm[-1])

In [ ]:
# TODO:
# Try attenuating the Hessian projection amount (interpolate down to no projection)